[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/skarma91/gen-ai-and-agentic-ai-course/blob/main/modules/module-1-python-for-ai/milestone-assignment/assignment.ipynb)

# Module 1 milestone: data facts, an LLM summary, saved to JSON

Put the whole module together in one small tool. It should:

1. **Load** a dataset (the CSV below) into a Pandas DataFrame.
2. **Compute** a few facts from it with Pandas (a type-hinted function returning a dict).
3. **Build a prompt** string from those facts.
4. **Ask an LLM** to summarize them (local Ollama; wrap the call in `try`/`except` so a missing service does not crash the tool).
5. **Save** the facts plus the summary to a JSON file, using a small class.

**Deliverable:** the working notebook and the JSON it writes. The LLM step needs Ollama running locally; without it, your `try`/`except` should fall back to a clear message so the rest still runs.

**Skills exercised:** functions, type hints, a class, files and JSON, Pandas, and an LLM call, everything from Module 1.

### The dataset

In [6]:
csv_text = """product,category,units,revenue
Widget,hardware,120,2400
Gadget,hardware,80,3200
Cable,accessory,300,1500
Case,accessory,150,1200
App,software,50,5000"""

import pandas as pd
from io import StringIO
df = pd.read_csv(StringIO(csv_text))
print(df)

  product   category  units  revenue
0  Widget   hardware    120     2400
1  Gadget   hardware     80     3200
2   Cable  accessory    300     1500
3    Case  accessory    150     1200
4     App   software     50     5000


### 1. Compute facts (Pandas)

In [20]:
# TODO: return a dict of a few facts about df, for example:
#   rows, total_revenue, top_category_by_revenue, avg_units
# Use groupby / sum / mean / idxmax as needed. Give it a type hint.
def compute_facts(df) -> dict:
    ...  # your code here
    return {
        "rows": int(len(df)),
        "total_revenue": int(df["revenue"].sum()),
        "top_category_by_revenue": df.groupby("category")["revenue"].mean().idxmax(),
        "avg_units": int(df["units"].mean())
    }

facts = compute_facts(df); print(facts)

{'rows': 5, 'total_revenue': 13300, 'top_category_by_revenue': 'software', 'avg_units': 140}


### 2. Build a prompt from the facts

In [26]:
import json
# TODO: return a prompt string that asks for a short summary and includes the facts.
def build_prompt(facts: dict) -> str:
    ...  # your code here
    prompt = f""" You are a master data analyst who has 20+ experince in data analysis\n
    your task is to analyze the datasets and provide a short summery for the given facts\n
    here's {facts} review the data and provide the outcome.
    """
    return prompt

build_prompt(facts)

" You are a master data analyst who has 20+ experince in data analysis\n\n    your task is to analyze the datasets and provide a short summery for the given facts\n\n    here's {'rows': 5, 'total_revenue': 13300, 'top_category_by_revenue': 'software', 'avg_units': 140} review the data and provide the outcome.\n    "

### 3. Ask the LLM (with a graceful fallback)

In [28]:
import requests
# TODO: POST the prompt to Ollama at http://localhost:11434/api/generate and return
# resp.json()["response"]. Wrap it in try/except so that if Ollama is not running you
# return a clear fallback message instead of crashing.
def ask_llm(prompt: str) -> str:
    ...  # your code here
    url = "http://localhost:11434/api/generate"
    body = {
        "model": "gemma4:12b",
        "prompt": prompt,
        "stream": False
    }
    try:
        resp = requests.post(url, json=body)
        summary = None
        if resp.status_code == 200:
            summary = resp.get("response")
            return summary
        else:
            raise f"Failed to fetch the response: {summary}"
    except Exception as e:
        raise f"Failed to connect with model: {e}"

### 4. Save facts + summary to JSON

In [ ]:
# TODO: a small class that saves {"facts": ..., "summary": ...} to a JSON file.
class Report:
    def __init__(self, path: str):
        self.path = path
    def save(self, facts: dict, summary: str) -> None:
        ...  # your code here
        with open("summary.json", 'w+') as file:
            file.write(json.dump(facts))
            file.write(json.dump(summary))

# Put it together:
facts = compute_facts(df)
summary = ask_llm(build_prompt(facts))
Report("report.json").save(facts, summary)